In [ ]:
import tensorflow as tf

In [ ]:
import numpy as np
from tensorflow import keras
import pandas as pd
from keras import layers

In [ ]:
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression

def evaluate_disentanglement(latents_pred, true_factors):
    n_latents = latents_pred.shape[1]
    n_factors = true_factors.shape[1]

    latents_centered = latents_pred - latents_pred.mean(axis=0)
    u, s, vh = np.linalg.svd(latents_centered, full_matrices=False)
    s_norm = s / s.sum()
    effective_rank = np.exp(-np.sum(s_norm * np.log(s_norm + 1e-10)))

    mi_matrix = np.zeros((n_factors, n_latents))
    for f in range(n_factors):
        mi_matrix[f, :] = mutual_info_regression(latents_pred, true_factors[:, f])

    mig_scores = []
    for f in range(n_factors):
        sorted_mi = np.sort(mi_matrix[f, :])[::-1]
        factor_entropy = mutual_info_regression(true_factors[:, f].reshape(-1, 1), true_factors[:, f])[0]
        gap = (sorted_mi[0] - sorted_mi[1]) / (factor_entropy + 1e-8)
        mig_scores.append(gap)
    mig = np.mean(mig_scores)


    importance_matrix = np.zeros((n_factors, n_latents))
    r2_scores = []
    for f in range(n_factors):
        rf = RandomForestRegressor(n_estimators=50, random_state=0)
        rf.fit(latents_pred, true_factors[:, f])
        importance_matrix[f, :] = rf.feature_importances_
        r2_scores.append(rf.score(latents_pred, true_factors[:, f]))

    latent_importance_norm = importance_matrix / (importance_matrix.sum(axis=0, keepdims=True) + 1e-8)
    entropy_per_latent = -np.sum(latent_importance_norm * np.log(latent_importance_norm + 1e-8), axis=0)
    disentanglement_per_latent = 1 - entropy_per_latent / np.log(n_factors)
    weight_per_latent = importance_matrix.sum(axis=0) / importance_matrix.sum()
    D = np.sum(disentanglement_per_latent * weight_per_latent)

    factor_importance_norm = importance_matrix / (importance_matrix.sum(axis=1, keepdims=True) + 1e-8)
    entropy_per_factor = -np.sum(factor_importance_norm * np.log(factor_importance_norm + 1e-8), axis=1)
    completeness_per_factor = 1 - entropy_per_factor / np.log(n_latents)
    C = np.mean(completeness_per_factor)

    I = np.mean(r2_scores)



In [ ]:
import numpy as np
import gc

FILE_PATH = 'dsprites_dataset_filepath'

with np.load(FILE_PATH, allow_pickle=True) as data:
    imgs_full = data['imgs']
    latents_classes = data['latents_classes']

max_pixel = imgs_full.max()

N_TOTAL = 20000
total_size = len(imgs_full)
idx = np.linspace(0, total_size - 1, N_TOTAL, dtype=int)

np.random.RandomState(42).shuffle(idx)

x_subset = imgs_full[idx]
factors_subset = latents_classes[idx]

del imgs_full, latents_classes, data
gc.collect()

if max_pixel > 1.0:
    x_subset = x_subset.astype("float32") / 255.0
else:
    x_subset = x_subset.astype("float32")

x_subset = np.expand_dims(x_subset, axis=-1)


x_train = x_subset[:18000]
x_test = x_subset[18000:]
factors_test = factors_subset[18000:]

del x_subset, factors_subset
gc.collect()




In [ ]:
import gc
import keras
import pandas as pd
import numpy as np
import tensorflow as tf


seeds = [0, 1, 2, 3, 4]
results = []

for seed in seeds:

    keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)


    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)
    h1 = layers.Dense(64, activation="relu")(x)
    h2 = layers.Dense(32, activation="relu")(h1)
    z = layers.Dense(10)(h2)
    d1 = layers.Dense(32, activation="relu")(z)
    d2 = layers.Dense(64, activation="relu")(d1)
    d_out = layers.Dense(64*64*1, activation="sigmoid")(d2)
    outputs = layers.Reshape((64, 64, 1))(d_out)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
    )

    encoder = keras.Model(inputs, z)
    latents_pred = encoder.predict(x_test, verbose=0)
    true_factors = factors_test

    mig, D, C, I, rank = evaluate_disentanglement(latents_pred, true_factors)
    
    results.append({
        "seed": seed,
        "val_loss": history.history["val_loss"][-1],
        "mig": mig, "D": D, "C": C, "I": I,
    })
    


df = pd.DataFrame(results)
print("\n=== AVERAGE RESULTS ===")
print(df.mean(numeric_only=True))
print(df.std(numeric_only=True))

#1d small no res

In [ ]:
seeds = [0, 1, 2, 3,4]
results = []

for seed in seeds:

    keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)

   

    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)
    
    h1 = layers.Dense(1024, activation="relu")(x)
    h2 = layers.Dense(512, activation="relu")(h1)
    h3 = layers.Dense(256, activation="relu")(h2)
    h4 = layers.Dense(128, activation="relu")(h3)
    h5 = layers.Dense(64, activation="relu")(h4)
    z = layers.Dense(10)(h5)   

    d1 = layers.Dense(64, activation="relu")(z)
    d2 = layers.Dense(128, activation="relu")(d1)
    d3 = layers.Dense(256, activation="relu")(d2)
    d4 = layers.Dense(512, activation="relu")(d3)
    d5 = layers.Dense(1024, activation="relu")(d4)
    d_out = layers.Dense(64*64*1, activation="sigmoid")(d5)
    outputs = layers.Reshape((64, 64, 1))(d_out)


    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
    )

    encoder = keras.Model(inputs, z)
    latents_pred = encoder.predict(x_test, verbose=0)
    true_factors = factors_test

    mig, D, C, I, rank = evaluate_disentanglement(latents_pred, true_factors)
    
    results.append({
        "seed": seed,
        "val_loss": history.history["val_loss"][-1],
        "mig": mig, "D": D, "C": C, "I": I,
    })
    


df = pd.DataFrame(results)
print("\n=== AVERAGE RESULTS ===")
print(df.mean(numeric_only=True))
print(df.std(numeric_only=True))



#1d big no res

In [ ]:
seeds = [0, 1, 2, 3,4]
results = []

for seed in seeds:

    keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)




    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)

    h1 = layers.Dense(64)(x)
    shortcut1 = layers.Dense(64)(x)
    h1 = layers.Add()([h1, shortcut1])
    h1 = layers.ReLU()(h1)

    h2 = layers.Dense(32)(h1)
    shortcut2 = layers.Dense(32)(h1)
    h2 = layers.Add()([h2, shortcut2])
    h2 = layers.ReLU()(h2)

    z = layers.Dense(10)(h2)   

    d1 = layers.Dense(32)(z)
    shortcut_d1 = layers.Dense(32)(z)
    d1 = layers.Add()([d1, shortcut_d1])
    d1 = layers.ReLU()(d1)

    d2 = layers.Dense(64)(d1)
    shortcut_d2 = layers.Dense(64)(d1)
    d2 = layers.Add()([d2, shortcut_d2])
    d2 = layers.ReLU()(d2)

    d_out = layers.Dense(64*64*1, activation="sigmoid")(d2)
    outputs = layers.Reshape((64, 64, 1))(d_out)


    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
    )

    encoder = keras.Model(inputs, z)
    latents_pred = encoder.predict(x_test, verbose=0)
    true_factors = factors_test

    mig, D, C, I, rank = evaluate_disentanglement(latents_pred, true_factors)
    
    results.append({
        "seed": seed,
        "val_loss": history.history["val_loss"][-1],
        "mig": mig, "D": D, "C": C, "I": I,
    })
    

df = pd.DataFrame(results)
print("\n=== AVERAGE RESULTS ===")
print(df.mean(numeric_only=True))
print(df.std(numeric_only=True))


#1d small with res

In [ ]:
seeds = [0, 1, 2, 3,4]
results = []

for seed in seeds:

    keras.backend.clear_session()
    gc.collect()

    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)




    inputs = keras.Input(shape=(64, 64, 1))
    x = layers.Flatten()(inputs)

    h1 = layers.Dense(1024)(x)
    shortcut1 = layers.Dense(1024)(x)
    h1 = layers.Add()([h1, shortcut1])
    h1 = layers.ReLU()(h1)
    
    h2 = layers.Dense(512)(h1)
    shortcut2 = layers.Dense(512)(h1)
    h2 = layers.Add()([h2, shortcut2])
    h2 = layers.ReLU()(h2)
    
    h3 = layers.Dense(256)(h2)
    shortcut3 = layers.Dense(256)(h2)
    h3 = layers.Add()([h3, shortcut3])
    h3 = layers.ReLU()(h3)
    
    h4 = layers.Dense(128)(h3)
    shortcut4 = layers.Dense(128)(h3)
    h4 = layers.Add()([h4, shortcut4])
    h4 = layers.ReLU()(h4)
    
    h5 = layers.Dense(64)(h4)
    shortcut5 = layers.Dense(64)(h4)
    h5 = layers.Add()([h5, shortcut5])
    h5 = layers.ReLU()(h5)
    
    z = layers.Dense(10)(h5)
    
    d1 = layers.Dense(64)(z)
    shortcut_d1 = layers.Dense(64)(z)
    d1 = layers.Add()([d1, shortcut_d1])
    d1 = layers.ReLU()(d1)
    
    d2 = layers.Dense(128)(d1)
    shortcut_d2 = layers.Dense(128)(d1)
    d2 = layers.Add()([d2, shortcut_d2])
    d2 = layers.ReLU()(d2)
    
    d3 = layers.Dense(256)(d2)
    shortcut_d3 = layers.Dense(256)(d2)
    d3 = layers.Add()([d3, shortcut_d3])
    d3 = layers.ReLU()(d3)
    
    d4 = layers.Dense(512)(d3)
    shortcut_d4 = layers.Dense(512)(d3)
    d4 = layers.Add()([d4, shortcut_d4])
    d4 = layers.ReLU()(d4)
    
    d5 = layers.Dense(1024)(d4)
    shortcut_d5 = layers.Dense(1024)(d4)
    d5 = layers.Add()([d5, shortcut_d5])
    d5 = layers.ReLU()(d5)

    d_out = layers.Dense(64*64*1, activation="sigmoid")(d5)
    outputs = layers.Reshape((64, 64, 1))(d_out)



    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    history = model.fit(
        x_train, x_train,
        validation_data=(x_test, x_test),
        epochs=50,
        batch_size=32,
        verbose=1
    )

    encoder = keras.Model(inputs, z)
    latents_pred = encoder.predict(x_test, verbose=0)
    true_factors = factors_test

    mig, D, C, I, rank = evaluate_disentanglement(latents_pred, true_factors)
    
    results.append({
        "seed": seed,
        "val_loss": history.history["val_loss"][-1],
        "mig": mig, "D": D, "C": C, "I": I,
    })
    


df = pd.DataFrame(results)
print("\n=== AVERAGE RESULTS ===")
print(df.mean(numeric_only=True))
print(df.std(numeric_only=True))


#1d big with res